IMPORT SECTION

In [ ]:
import os
import random
import glob
import json
from IPython.display import HTML, display
from utils.utils import preprocess_libero_dataset, demo_animator 
import ipywidgets as widgets
from IPython.display import display
from Dataset.LiberoDataset import LiberoDataset
from model.ActionJEPA import ActionJEPA
from model.modules.CLIPEncoder import CLIPEncoder
from model.modules.VJEPAEncoder import VJEPAEncoder
from training.train import train
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import numpy as np


# REPRODUCIBILITY 
seed = 46

# Set seed for torch, numpy and random libraries
torch.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(seed)
random.seed(seed)

# Set the devide mode on GPU (if available CUDA for Nvidia and  MPS for Apple Silicon) or CPU
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

In [ ]:
# Loading the config.json file
with open('config.json', 'r') as f:
    config = json.load(f)

# Hyperparameters definition
NUM_FRAMES = config['num_frames']
NUM_EPOCHS = config['num_epochs']
BATCH_SIZE = config['batch_size']
LEARNING_RATE = config['learning_rate']
NUM_WORKERS = config['num_workers']
LAMBDA_ACTOR = config['lambda_actor']
LAMBDA_REFINER = config['lambda_refiner']


In [ ]:
ALL_AVAILABLE_TASKS = ["libero_10", "libero_90", "libero_goal", "libero_object", "libero_spatial"]

task_selector = widgets.SelectMultiple(
    options=ALL_AVAILABLE_TASKS,
    value=["libero_10"], # default selection
    description='Select:',
    style={'description_width': 'initial'},
    layout={'width': '350px', 'height': '150px'}
)

print("Select the tasks you want to process/load and to use for training (Hold Ctrl or Cmd for multiple):")
display(task_selector)

selected_tasks = list(task_selector.value)

In [ ]:
# Path for the VJEPA Vision Encoder
vjepa_path = "checkpoints/facebook/vjepa2-vitg-fpc64-256"

vision_backbone = VJEPAEncoder(
    model_path=vjepa_path,
    frozen=True,
    device=device
).to(device)

# Path for the CLIP Language Encoder 
clip_path = "checkpoints/openai/clip-vit-large-patch14"

language_backbone = CLIPEncoder(
    model_path=clip_path,
    frozen=True,
    device=device
).to(device)

In [ ]:
libero_paths = [f"./LIBERO/datasets/{task}/*.hdf5" for task in selected_tasks]
processed_data_dir ='./processed_data' 

for path in libero_paths:
    preprocess_libero_dataset(hdf5_path=path, 
                              output_dir=processed_data_dir,
                              vision_backbone=vision_backbone,
                              language_backbone=language_backbone,
                              use_backbone=True,
                              num_frames = NUM_FRAMES
                              ) 

In [ ]:
data_dir = './processed_data/libero_10/'
pattern = os.path.join(data_dir, "*pt")
all_demo_paths = glob.glob(pattern)

if not all_demo_paths:
    raise FileNotFoundError(f"No demo file founded in {data_dir}. Please process the data with the previous cell!")
else:
    random_demo_path = random.choice(all_demo_paths)
    ani = demo_animator(demo_pt_path=random_demo_path)
    display(HTML(ani.to_jshtml()))

In [ ]:
dataset = LiberoDataset(data_dir='./processed_data', 
                        selected_tasks=selected_tasks, 
                        window_size=NUM_FRAMES, 
                        stride=1,
                        use_features = True
                        )

train_percentage = 0.7
val_percentage = 0.2

train_size = int(train_percentage*len(dataset))
val_size = int(val_percentage*len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset=dataset, lengths=[train_size, val_size, test_size])

print(f"Total dataset size (chunks): {len(dataset)}, Train Size (chunks): {len(train_dataset)}, Validation Size (chunks): {len(val_dataset)}, Test Size (steps): {len(test_dataset)}")

In [ ]:
# Path for all the models
vjepa_path = "checkpoints/facebook/vjepa2-vitg-fpc64-256"
vjepa_pred_path = "checkpoints/facebook/jepa-wms/vjepa2_ac_droid.pth.tar/vjepa2_ac_droid.pth.tar"
clip_path = "checkpoints/openai/clip-vit-large-patch14"

model = ActionJEPA(
    vjepa_encoder_path=vjepa_path,
    vjepa_predictor_path=vjepa_pred_path,
    clip_model_path=clip_path,
    num_frames=NUM_FRAMES,
    use_backbone = False,
    device=device
).to(device)

model.print_model_info()

In [ ]:
# Name of the directory for the results
results_dir_path = "./results"
os.makedirs(results_dir_path, exist_ok=True)

# Definition of the Categorical Cross Entropy Loss
loss_fn = nn.MSELoss()
# Definition of the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr = LEARNING_RATE)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=NUM_WORKERS, 
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS, 
    pin_memory=True
)

train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    num_epochs=NUM_EPOCHS,
    config=config,
    device=device,
    results_dir_path=results_dir_path,
    lambda_actor=LAMBDA_ACTOR,
    lambda_refiner=LAMBDA_REFINER

)